# 05 · Enfoque de asignación de primas y pasivo por siniestros incurridos

**LRC (par. 55).** Es igual a las primas recibidas menos el ingreso reconocido y menos los gastos de adquisición aún no amortizados.

- Autos y TRC difieren la adquisición; incendio/hogar la reconoce como gasto (59(a)).
- El ingreso sigue el paso del tiempo (B126(a)); en TRC sigue el patrón esperado de siniestros, porque la exposición crece con la obra (B126(b)).
- El LRC no se ajusta por financiación, porque entre prima y servicio no pasa más de un año (par. 56).

**Prueba de onerosidad (par. 57–58).** Compara los flujos de cumplimiento de la cobertura restante con ese saldo.

**LIC.** Suma tres partes:

- las reservas de caso de los siniestros reportados;
- un IBNR tipo Bornhuetter-Ferguson: siniestros esperados de cada mes de ocurrencia por la proporción aún no reportada;
- los gastos de liquidación (B65(f)).

Autos y TRC se descuentan; incendio/hogar no, porque paga dentro del año (59(b)). El ajuste por riesgo es el percentil 75 de una lognormal, con diversificación entre coberturas.

In [1]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, '../src')
pd.set_option('display.float_format', lambda x: f'{x:,.0f}' if abs(x) >= 100 else f'{x:,.4f}')
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 200)
from ifrs17 import closing as C, pc, disclosures as D
from ifrs17.data import load
res = C.run()
pd.concat(res['paa']['lrc'])[['closing', 'group', 'premium_written', 'premiums_received_cum', 'revenue_cum', 'lrc_excl_lc', 'fcf_remaining', 'loss_component']]

,closing,group,premium_written,premiums_received_cum,revenue_cum,lrc_excl_lc,fcf_remaining,loss_component
0,2023-12-31,AUTO-2023-RES,"1,355,160","1,196,666","687,485","432,804","380,104",0.0000
1,2023-12-31,AUTO-2023-SIG,"10,851,928","9,616,896","5,438,467","3,551,667","2,785,413",0.0000
2,2023-12-31,INC-2023-SIG,"3,993,772","3,993,772","2,013,571","1,980,201","1,291,966",0.0000
3,2023-12-31,TRC-2023-SIG,"3,463,890","3,463,890","107,019","2,954,047","2,139,081",0.0000
0,2024-12-31,AUTO-2023-RES,"1,355,160","1,355,159","1,355,160",0.1225,0.0000,0.0000
1,2024-12-31,AUTO-2023-SIG,"10,851,928","10,851,928","10,851,928",2.7745,0.0000,0.0000
2,2024-12-31,AUTO-2024-RES,"1,379,616","1,217,930","684,057","453,791","472,906","19,115"
3,2024-12-31,AUTO-2024-SIG,"11,088,704","9,825,049","5,560,102","3,625,206","2,863,879",0.0000
4,2024-12-31,INC-2023-SIG,"3,993,772","3,993,772","3,993,772",0.0000,0.0000,0.0000
5,2024-12-31,INC-2024-SIG,"4,068,924","4,068,924","2,068,558","2,000,367","1,313,058",0.0000


Al cierre de 2024, AUTO-2024-RES (flotas) tiene un componente de pérdida. Con el índice de repuestos ya conocido, los siniestros esperados de la cobertura que falta más el ajuste por riesgo y los gastos superan el LRC (par. 57–58). Al año siguiente la cobertura terminó y el componente se libera. AUTO-2025-ONE nace onerosa.

## LIC: reservas de caso más IBNR

Como la verdad simulada incluye los pagos posteriores, se puede comparar la estimación de cada cierre con lo que de verdad se pagó después (retrospectiva).

In [2]:
filas = []
for c, lic in zip(['2023-12-31', '2024-12-31', '2025-12-31'], res['paa']['lic']):
    for linea in pc.LINES:
        l = lic[lic['line'] == linea]
        t = pc.LINES[linea]['table']
        claims = load(f'{t}_claims'); pay = load(f'{t}_payments').merge(claims[['claim_id', 'occurrence_date']], on='claim_id')
        real = pay[(pay['occurrence_date'] <= c) & (pay['payment_date'] > c)]['amount'].sum()
        filas.append({'cierre': c, 'línea': linea, 'casos': l['case'].sum(), 'IBNR': l['ibnr'].sum(), 'estimación': l['case'].sum() + l['ibnr'].sum(),
                      'pagado_después': real, 'LIC_descontado': l['pv'].sum(), 'ajuste_por_riesgo': l['ra'].sum()})
pd.DataFrame(filas)

,cierre,línea,casos,IBNR,estimación,pagado_después,LIC_descontado,ajuste_por_riesgo
0,2023-12-31,AUTO,"1,598,960","973,544","2,572,504","2,602,012","2,503,681","255,556"
1,2023-12-31,INC,"138,612","150,648","289,261","247,796","297,938","40,811"
2,2023-12-31,TRC,0.0000,"22,717","22,717","12,921","22,610","3,770"
3,2024-12-31,AUTO,"3,963,797","1,288,484","5,252,281","5,547,642","5,091,437","439,750"
4,2024-12-31,INC,"509,334","171,448","680,782","712,106","701,205","90,049"
5,2024-12-31,TRC,"457,785","120,218","578,004","1,167,614","581,154","90,858"
6,2025-12-31,AUTO,"4,942,783","1,332,662","6,275,445","6,153,680","6,129,702","523,378"
7,2025-12-31,INC,"444,380","175,522","619,902","592,439","638,499","80,910"
8,2025-12-31,TRC,"782,385","281,847","1,064,232","895,352","1,073,902","165,670"


La estimación de autos queda cerca de lo que después se pagó. En TRC las diferencias son grandes porque hay pocas obras y siniestros de cola pesada: es justo la incertidumbre que el ajuste por riesgo compensa. El nivel de confianza equivalente del ajuste por riesgo de daños, ya diversificado, se revela según el par. 119.

In [3]:
D.confidence_levels(res)

,portfolio,technique,risk_adjustment,capital_99_5,confidence_level
0,Vida temporal,costo de capital 6%,"3,413,217","7,491,576",0.8797
1,Daños (LIC),"percentil 75, lognormal, correlación 25%","769,958",NaN,0.7327


## Desarrollo de siniestros de autos (par. 130)

In [4]:
D.claims_development('AUTO')

,closing,accident_year,paid_to_date,outstanding_undiscounted,lic_pv,ultimate_estimate
0,2023-12-31,2023,"1,401,589","2,649,679","2,503,681","4,051,268"
1,2024-12-31,2023,"3,348,181","616,106","574,506","3,964,287"
2,2024-12-31,2024,"4,009,361","4,793,744","4,516,931","8,803,105"
3,2025-12-31,2023,"3,701,038","311,704","294,355","4,012,742"
4,2025-12-31,2024,"7,215,936","1,761,853","1,659,989","8,977,789"
5,2025-12-31,2025,"4,441,341","4,390,151","4,175,359","8,831,492"


![Desarrollo](../docs/figures/desarrollo_autos.png)

La reconciliación con el LIC es directa: el pendiente no descontado menos el efecto del descuento es el valor presente del LIC, y a él se suma el ajuste por riesgo.